# Laboratorio 12 — Series Temporales
**Curso:** Minería de Datos (EIN132A25)

## Objetivos
- Cargar y manipular series temporales con **pandas**
- Identificar **tendencia** y **estacionalidad**
- Aplicar **rolling statistics**
- Construir un modelo simple de **forecasting**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

np.random.seed(42)
fechas = pd.date_range(start="2020-01-01", end="2023-12-31", freq="D")
tendencia  = np.linspace(100, 200, len(fechas))
estacional = 30 * np.sin(2 * np.pi * np.arange(len(fechas)) / 365)
ruido      = np.random.normal(0, 10, len(fechas))
ventas     = tendencia + estacional + ruido

df = pd.DataFrame({"fecha": fechas, "ventas": ventas})
df = df.set_index("fecha")
print(df.head())
print(f"Tipo del índice: {type(df.index)}")

## 1. Selección por fecha y resampleo

In [ ]:
df_2022 = df["2022"]
print(f"Registros en 2022: {len(df_2022)}")

df_mensual   = df.resample("ME").mean()
df_semanal   = df.resample("W").sum()
df_trimestral = df.resample("QE").mean()
print(f"Mensual: {len(df_mensual)} filas")

## 2. Visualización en distintas frecuencias

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].plot(df.index, df["ventas"], color="steelblue", linewidth=0.8, alpha=0.8)
axes[0].set_title("Ventas diarias")

axes[1].plot(df_mensual.index, df_mensual["ventas"], color="salmon",
             linewidth=2, marker="o", markersize=4)
axes[1].set_title("Ventas mensuales (promedio)")

axes[2].bar(df_trimestral.index, df_trimestral["ventas"],
            color="steelblue", alpha=0.7, width=60)
axes[2].set_title("Ventas trimestrales")

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.set_ylabel("Ventas")

plt.tight_layout()
plt.show()

## 3. Rolling statistics

In [ ]:
ventana = 30
df["media_movil"] = df["ventas"].rolling(window=ventana).mean()
df["std_movil"]   = df["ventas"].rolling(window=ventana).std()

plt.figure(figsize=(14, 6))
plt.plot(df.index, df["ventas"], alpha=0.4, color="steelblue", label="Ventas diarias")
plt.plot(df.index, df["media_movil"], color="red", linewidth=2, label=f"Media móvil ({ventana}d)")
plt.fill_between(df.index,
    df["media_movil"] - df["std_movil"],
    df["media_movil"] + df["std_movil"],
    alpha=0.2, color="red", label="±1 std")
plt.title("Ventas con media móvil")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Análisis por período

In [ ]:
df["mes"]        = df.index.month
df["dia_semana"] = df.index.dayofweek

ventas_por_mes = df.groupby("mes")["ventas"].mean()
ventas_por_mes.plot(kind="bar", color="steelblue", edgecolor="black", figsize=(10, 5))
plt.title("Ventas promedio por mes")
plt.xticks(range(12), ["Ene","Feb","Mar","Abr","May","Jun",
                        "Jul","Ago","Sep","Oct","Nov","Dic"], rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()

## 5. Descomposición de la serie

In [ ]:
result = seasonal_decompose(df_mensual["ventas"], model="additive", period=12)

fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(14, 10))
result.observed.plot(ax=ax1, color="steelblue"); ax1.set_title("Serie original")
result.trend.plot(ax=ax2, color="red"); ax2.set_title("Tendencia")
result.seasonal.plot(ax=ax3, color="green"); ax3.set_title("Estacionalidad")
result.resid.plot(ax=ax4, color="orange"); ax4.set_title("Residuo")
for ax in [ax1, ax2, ax3, ax4]:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Forecasting con regresión lineal

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

df_m = df_mensual.copy().reset_index()
df_m["t"] = np.arange(len(df_m))
df_m["mes_sin"] = np.sin(2 * np.pi * df_m["fecha"].dt.month / 12)
df_m["mes_cos"] = np.cos(2 * np.pi * df_m["fecha"].dt.month / 12)

n_test = 6
X_ts = df_m[["t", "mes_sin", "mes_cos"]]
y_ts = df_m["ventas"]
X_tr, X_te = X_ts[:-n_test], X_ts[-n_test:]
y_tr, y_te = y_ts[:-n_test], y_ts[-n_test:]

modelo = LinearRegression()
modelo.fit(X_tr, y_tr)
y_pred = modelo.predict(X_te)

mae  = mean_absolute_error(y_te, y_pred)
rmse = np.sqrt(mean_squared_error(y_te, y_pred))
mape = np.mean(np.abs((y_te.values - y_pred) / y_te.values)) * 100
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.1f}%")

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df_m["fecha"][:-n_test], y_tr, color="steelblue", label="Train")
plt.plot(df_m["fecha"][-n_test:], y_te, color="green", label="Test real", linewidth=2)
plt.plot(df_m["fecha"][-n_test:], y_pred, color="red",
         linestyle="--", linewidth=2, label="Predicción")
plt.title("Forecasting con Regresión Lineal")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Ejercicios

### Ejercicio 1 — Media móvil con distintas ventanas

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df.index, df["ventas"], alpha=0.3, color="steelblue", label="Original")
for ventana, color in [(7, "red"), (30, "green"), (90, "purple")]:
    plt.plot(df.index, df["ventas"].rolling(ventana).mean(),
             label=f"MM {ventana}d", linewidth=1.5)
plt.title("Comparación de medias móviles")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Desafío — Forecasting con features lag

In [ ]:
df_lag = df_m.copy()
df_lag["lag_1"]  = df_lag["ventas"].shift(1)
df_lag["lag_12"] = df_lag["ventas"].shift(12)
df_lag = df_lag.dropna()

features_lag = ["t", "mes_sin", "mes_cos", "lag_1", "lag_12"]
X_lag = df_lag[features_lag]
y_lag = df_lag["ventas"]
X_lag_tr, X_lag_te = X_lag[:-n_test], X_lag[-n_test:]
y_lag_tr, y_lag_te = y_lag[:-n_test], y_lag[-n_test:]

m_lag = LinearRegression()
m_lag.fit(X_lag_tr, y_lag_tr)
y_lag_pred = m_lag.predict(X_lag_te)

mae_lag = mean_absolute_error(y_lag_te, y_lag_pred)
mape_lag = np.mean(np.abs((y_lag_te.values - y_lag_pred) / y_lag_te.values)) * 100
print(f"Con lags — MAE: {mae_lag:.2f}, MAPE: {mape_lag:.1f}%")